In [1]:
# =========================
# STEP 1: IMPORT LIBRARIES
# =========================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# =========================================
# STEP 2: DOWNLOAD OFFICIAL CHICAGO DATA
# =========================================

crime_url = "https://data.cityofchicago.org/resource/t7ek-mgzi.csv?$limit=50000"
arrests_url = "https://data.cityofchicago.org/resource/dpt3-jri9.csv?$limit=50000"

crime_raw = pd.read_csv(crime_url)
arrests_raw = pd.read_csv(arrests_url)

print("Crime raw shape:", crime_raw.shape)
print("Arrests raw shape:", arrests_raw.shape)

crime_raw.head()

Crime raw shape: (50000, 22)
Arrests raw shape: (50000, 24)


,id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_area,fbi_code,x_coordinate,y_coordinate,year,updated_on,latitude,longitude,location
0,14075483,JK105557,2025-12-31T23:58:00.000,050XX S PAULINA ST,0560,ASSAULT,SIMPLE,RESIDENCE,False,False,931,9,20,61.0,08A,1165860.0,1871343.0,2025,2026-01-09T15:40:47.000,41.802549,-87.667246,POINT (-87.667246428 41.802549018)
1,14070833,JK100050,2025-12-31T23:55:00.000,053XX W WASHINGTON BLVD,0930,MOTOR VEHICLE THEFT,THEFT / RECOVERY - AUTOMOBILE,APARTMENT,False,True,1522,15,37,25.0,07,1140809.0,1900235.0,2025,2026-01-08T15:46:48.000,41.882329,-87.758411,POINT (-87.758411303 41.882328854)
2,14070745,JK100011,2025-12-31T23:54:00.000,100XX W OHARE ST,2890,PUBLIC PEACE VIOLATION,OTHER VIOLATION,AIRCRAFT,False,False,1651,16,41,76.0,24,1100658.0,1934241.0,2025,2026-01-08T15:46:48.000,41.976290,-87.905227,POINT (-87.905227221 41.976290414)
3,14070799,JK100014,2025-12-31T23:54:00.000,100XX W OHARE ST,2890,PUBLIC PEACE VIOLATION,OTHER VIOLATION,AIRCRAFT,False,False,1651,16,41,76.0,24,1100658.0,1934241.0,2025,2026-01-08T15:46:48.000,41.976290,-87.905227,POINT (-87.905227221 41.976290414)
4,14070845,JK100006,2025-12-31T23:54:00.000,013XX W LAKE ST,0454,BATTERY,"AGGRAVATED P.O. - HANDS, FISTS, FEET, NO / MIN...",RESTAURANT,True,False,1215,12,27,28.0,08B,1167120.0,1901555.0,2025,2026-01-08T15:46:48.000,41.885427,-87.661759,POINT (-87.661759042 41.885426714)


In [3]:
# =========================================
# STEP 3: SEE THE COLUMNS
# =========================================

print("Crime columns:")
print(list(crime_raw.columns))

print("\nArrests columns:")
print(list(arrests_raw.columns))

Crime columns:
['id', 'case_number', 'date', 'block', 'iucr', 'primary_type', 'description', 'location_description', 'arrest', 'domestic', 'beat', 'district', 'ward', 'community_area', 'fbi_code', 'x_coordinate', 'y_coordinate', 'year', 'updated_on', 'latitude', 'longitude', 'location']

Arrests columns:
['cb_no', 'case_number', 'arrest_date', 'race', 'charge_1_statute', 'charge_1_description', 'charge_1_type', 'charge_1_class', 'charge_2_statute', 'charge_2_description', 'charge_2_type', 'charge_2_class', 'charge_3_statute', 'charge_3_description', 'charge_3_type', 'charge_3_class', 'charge_4_statute', 'charge_4_description', 'charge_4_type', 'charge_4_class', 'charges_statute', 'charges_description', 'charges_type', 'charges_class']


In [4]:
# =========================================
# STEP 4: CREATE PROJECT FOLDERS
# =========================================

base_path = "/content/public_safety_project"

folders = [
    "data_raw",
    "data_clean",
    "reports"
]

for folder in folders:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [5]:
# =========================================
# STEP 5: SAVE RAW FILES
# =========================================

crime_raw.to_csv(f"{base_path}/data_raw/crime_incidents_raw.csv", index=False)
arrests_raw.to_csv(f"{base_path}/data_raw/arrests_raw.csv", index=False)

print("Raw files saved.")

Raw files saved.


In [6]:
# =========================================
# STEP 6: CLEAN CRIME DATA
# =========================================

crime = crime_raw.copy()

# Standardize column names
crime.columns = crime.columns.str.strip().str.lower().str.replace(" ", "_")

# Convert date columns if present
if "date" in crime.columns:
    crime["date"] = pd.to_datetime(crime["date"], errors="coerce")

# Keep only useful columns if they exist
crime_keep_cols = [
    "id",
    "date",
    "primary_type",
    "description",
    "location_description",
    "arrest",
    "domestic",
    "district",
    "beat",
    "ward",
    "community_area",
    "year",
    "latitude",
    "longitude"
]

crime = crime[[col for col in crime_keep_cols if col in crime.columns]]

# Remove rows without date
if "date" in crime.columns:
    crime = crime.dropna(subset=["date"])

# Create useful derived columns
crime["incident_date"] = crime["date"].dt.date
crime["incident_year"] = crime["date"].dt.year
crime["incident_month"] = crime["date"].dt.month
crime["incident_month_name"] = crime["date"].dt.month_name()
crime["incident_day"] = crime["date"].dt.day
crime["incident_hour"] = crime["date"].dt.hour
crime["incident_weekday"] = crime["date"].dt.day_name()

# Fill some common missing text fields
for col in ["primary_type", "description", "location_description", "district"]:
    if col in crime.columns:
        crime[col] = crime[col].fillna("Unknown")

print("Crime cleaned shape:", crime.shape)
crime.head()

Crime cleaned shape: (50000, 21)


,id,date,primary_type,description,location_description,arrest,domestic,district,beat,ward,community_area,year,latitude,longitude,incident_date,incident_year,incident_month,incident_month_name,incident_day,incident_hour,incident_weekday
0,14075483,2025-12-31 23:58:00,ASSAULT,SIMPLE,RESIDENCE,False,False,9,931,20,61.0,2025,41.802549,-87.667246,2025-12-31,2025,12,December,31,23,Wednesday
1,14070833,2025-12-31 23:55:00,MOTOR VEHICLE THEFT,THEFT / RECOVERY - AUTOMOBILE,APARTMENT,False,True,15,1522,37,25.0,2025,41.882329,-87.758411,2025-12-31,2025,12,December,31,23,Wednesday
2,14070745,2025-12-31 23:54:00,PUBLIC PEACE VIOLATION,OTHER VIOLATION,AIRCRAFT,False,False,16,1651,41,76.0,2025,41.976290,-87.905227,2025-12-31,2025,12,December,31,23,Wednesday
3,14070799,2025-12-31 23:54:00,PUBLIC PEACE VIOLATION,OTHER VIOLATION,AIRCRAFT,False,False,16,1651,41,76.0,2025,41.976290,-87.905227,2025-12-31,2025,12,December,31,23,Wednesday
4,14070845,2025-12-31 23:54:00,BATTERY,"AGGRAVATED P.O. - HANDS, FISTS, FEET, NO / MIN...",RESTAURANT,True,False,12,1215,27,28.0,2025,41.885427,-87.661759,2025-12-31,2025,12,December,31,23,Wednesday


In [7]:
# =========================================
# STEP 7: SAVE CLEANED CRIME DATA
# =========================================

crime.to_csv(f"{base_path}/data_clean/crime_incidents_clean.csv", index=False)
print("Cleaned crime data saved.")

Cleaned crime data saved.


In [8]:
# =========================================
# STEP 8: CLEAN ARRESTS DATA
# =========================================

arrests = arrests_raw.copy()

# Standardize column names
arrests.columns = arrests.columns.str.strip().str.lower().str.replace(" ", "_")

# Convert arrest date if present
if "arrest_date" in arrests.columns:
    arrests["arrest_date"] = pd.to_datetime(arrests["arrest_date"], errors="coerce")

# Keep useful columns
arrests_keep_cols = [
    "id",
    "arrest_date",
    "arrest_type",
    "charge_1",
    "city",
    "state",
    "zip",
    "charge_count",
    "rd_no",
    "age",
    "gender",
    "race",
    "district",
    "beat"
]

arrests = arrests[[col for col in arrests_keep_cols if col in arrests.columns]]

# Drop rows with no arrest date
if "arrest_date" in arrests.columns:
    arrests = arrests.dropna(subset=["arrest_date"])

# Create derived columns
arrests["arrest_day"] = arrests["arrest_date"].dt.day
arrests["arrest_month"] = arrests["arrest_date"].dt.month
arrests["arrest_month_name"] = arrests["arrest_date"].dt.month_name()
arrests["arrest_year"] = arrests["arrest_date"].dt.year
arrests["arrest_hour"] = arrests["arrest_date"].dt.hour
arrests["arrest_weekday"] = arrests["arrest_date"].dt.day_name()

# Fill missing text columns
for col in ["charge_1", "gender", "race", "district"]:
    if col in arrests.columns:
        arrests[col] = arrests[col].fillna("Unknown")

print("Arrests cleaned shape:", arrests.shape)
arrests.head()

Arrests cleaned shape: (50000, 8)


,arrest_date,race,arrest_day,arrest_month,arrest_month_name,arrest_year,arrest_hour,arrest_weekday
0,2026-03-07 23:06:00,BLACK,7,3,March,2026,23,Saturday
1,2026-03-07 23:45:00,WHITE,7,3,March,2026,23,Saturday
2,2026-03-07 22:29:00,ASIAN / PACIFIC ISLANDER,7,3,March,2026,22,Saturday
3,2026-03-07 23:51:00,WHITE HISPANIC,7,3,March,2026,23,Saturday
4,2026-03-07 23:57:00,BLACK HISPANIC,7,3,March,2026,23,Saturday


In [9]:
# =========================================
# STEP 9: SAVE CLEANED ARRESTS DATA
# =========================================

arrests.to_csv(f"{base_path}/data_clean/arrests_clean.csv", index=False)
print("Cleaned arrests data saved.")

Cleaned arrests data saved.


In [10]:
# =========================================
# STEP 10: GENERATE JAIL OPERATIONS DATA
# =========================================

np.random.seed(42)
random.seed(42)

start_date = datetime(2025, 1, 1)
num_days = 180

facilities = [
    {"facility_name": "North Jail Facility", "capacity": 850},
    {"facility_name": "South Jail Facility", "capacity": 700}
]

jail_rows = []

for facility in facilities:
    base_population = int(facility["capacity"] * 0.88)

    for i in range(num_days):
        current_date = start_date + timedelta(days=i)

        seasonal_effect = int(10 * np.sin(i / 12))
        random_variation = random.randint(-15, 15)

        jail_population = base_population + seasonal_effect + random_variation
        jail_population = max(jail_population, int(facility["capacity"] * 0.75))
        jail_population = min(jail_population, int(facility["capacity"] * 1.05))

        bookings = random.randint(18, 40)
        releases = random.randint(15, 35)
        staff_on_duty = random.randint(75, 110) if facility["capacity"] > 800 else random.randint(60, 95)
        critical_incidents = random.randint(0, 5)
        maintenance_requests = random.randint(2, 12)
        medical_watch_inmates = random.randint(8, 25)

        jail_rows.append({
            "report_date": current_date.date(),
            "facility_name": facility["facility_name"],
            "jail_population": jail_population,
            "capacity": facility["capacity"],
            "bookings": bookings,
            "releases": releases,
            "staff_on_duty": staff_on_duty,
            "critical_incidents": critical_incidents,
            "maintenance_requests": maintenance_requests,
            "medical_watch_inmates": medical_watch_inmates
        })

jail_operations = pd.DataFrame(jail_rows)

# Derived fields
jail_operations["capacity_utilization_pct"] = round(
    (jail_operations["jail_population"] / jail_operations["capacity"]) * 100, 2
)

jail_operations["net_change"] = jail_operations["bookings"] - jail_operations["releases"]

print("Jail operations shape:", jail_operations.shape)
jail_operations.head()

Jail operations shape: (360, 12)


,report_date,facility_name,jail_population,capacity,bookings,releases,staff_on_duty,critical_incidents,maintenance_requests,medical_watch_inmates,capacity_utilization_pct,net_change
0,2025-01-01,North Jail Facility,753,850,21,15,92,1,5,12,88.59,6
1,2025-01-02,North Jail Facility,756,850,21,32,80,4,8,9,88.94,-11
2,2025-01-03,North Jail Facility,734,850,20,21,89,4,11,8,86.35,-1
3,2025-01-04,North Jail Facility,752,850,24,35,109,3,5,22,88.47,-11
4,2025-01-05,North Jail Facility,754,850,26,15,85,5,8,18,88.71,11


In [11]:
# =========================================
# STEP 11: SAVE JAIL OPERATIONS DATA
# =========================================

jail_operations.to_csv(f"{base_path}/data_clean/jail_operations.csv", index=False)
jail_operations.to_csv(f"{base_path}/data_raw/jail_operations.csv", index=False)

print("Jail operations data saved.")

Jail operations data saved.


In [12]:
# =========================================
# STEP 12: GENERATE OPERATIONS SUPPORT DATA
# =========================================

ops_rows = []

for facility in facilities:
    for i in range(num_days):
        current_date = start_date + timedelta(days=i)

        open_posts = random.randint(4, 18)
        overtime_hours = random.randint(25, 120)
        maintenance_backlog = random.randint(5, 30)
        work_orders_closed = random.randint(1, 10)
        transport_runs = random.randint(2, 12)
        special_housing_inmates = random.randint(10, 35)

        ops_rows.append({
            "report_date": current_date.date(),
            "facility_name": facility["facility_name"],
            "open_posts": open_posts,
            "overtime_hours": overtime_hours,
            "maintenance_backlog": maintenance_backlog,
            "work_orders_closed": work_orders_closed,
            "transport_runs": transport_runs,
            "special_housing_inmates": special_housing_inmates
        })

operations_support = pd.DataFrame(ops_rows)

print("Operations support shape:", operations_support.shape)
operations_support.head()

Operations support shape: (360, 8)


,report_date,facility_name,open_posts,overtime_hours,maintenance_backlog,work_orders_closed,transport_runs,special_housing_inmates
0,2025-01-01,North Jail Facility,8,108,16,4,2,31
1,2025-01-02,North Jail Facility,5,84,14,3,8,31
2,2025-01-03,North Jail Facility,12,115,29,5,3,30
3,2025-01-04,North Jail Facility,18,62,16,10,5,17
4,2025-01-05,North Jail Facility,6,86,9,8,11,21


In [13]:
# =========================================
# STEP 13: SAVE OPERATIONS SUPPORT DATA
# =========================================

operations_support.to_csv(f"{base_path}/data_clean/operations_support.csv", index=False)
operations_support.to_csv(f"{base_path}/data_raw/operations_support.csv", index=False)

print("Operations support data saved.")

Operations support data saved.


In [14]:
# =========================================
# STEP 14: CREATE A RISK / RESOURCE GAP TABLE
# =========================================

risk_df = jail_operations.merge(
    operations_support,
    on=["report_date", "facility_name"],
    how="left"
)

def assign_risk(row):
    if row["capacity_utilization_pct"] > 100:
        return "Over Capacity"
    elif row["open_posts"] >= 15:
        return "Staffing Gap"
    elif row["maintenance_backlog"] >= 25:
        return "Facility Risk"
    elif row["critical_incidents"] >= 4:
        return "Incident Spike"
    else:
        return "Normal"

risk_df["risk_flag"] = risk_df.apply(assign_risk, axis=1)

risk_df.head()

,report_date,facility_name,jail_population,capacity,bookings,releases,staff_on_duty,critical_incidents,maintenance_requests,medical_watch_inmates,capacity_utilization_pct,net_change,open_posts,overtime_hours,maintenance_backlog,work_orders_closed,transport_runs,special_housing_inmates,risk_flag
0,2025-01-01,North Jail Facility,753,850,21,15,92,1,5,12,88.59,6,8,108,16,4,2,31,Normal
1,2025-01-02,North Jail Facility,756,850,21,32,80,4,8,9,88.94,-11,5,84,14,3,8,31,Incident Spike
2,2025-01-03,North Jail Facility,734,850,20,21,89,4,11,8,86.35,-1,12,115,29,5,3,30,Facility Risk
3,2025-01-04,North Jail Facility,752,850,24,35,109,3,5,22,88.47,-11,18,62,16,10,5,17,Staffing Gap
4,2025-01-05,North Jail Facility,754,850,26,15,85,5,8,18,88.71,11,6,86,9,8,11,21,Incident Spike


In [15]:
# =========================================
# STEP 15: SAVE RISK TABLE
# =========================================

risk_df.to_csv(f"{base_path}/data_clean/risk_analysis_table.csv", index=False)
print("Risk table saved.")

Risk table saved.


In [16]:
# =========================================
# STEP 16: MONTHLY CRIME SUMMARY
# =========================================

monthly_crime = crime.groupby(["incident_year", "incident_month", "incident_month_name"]).size().reset_index(name="total_incidents")
monthly_crime = monthly_crime.sort_values(["incident_year", "incident_month"])

monthly_crime.to_csv(f"{base_path}/reports/monthly_crime_summary.csv", index=False)

monthly_crime.head()

,incident_year,incident_month,incident_month_name,total_incidents
0,2025,10,October,14343
1,2025,11,November,18317
2,2025,12,December,17340


In [17]:
# =========================================
# STEP 17: CRIME BY TYPE SUMMARY
# =========================================

crime_by_type = crime.groupby("primary_type").size().reset_index(name="total_incidents").sort_values("total_incidents", ascending=False)

crime_by_type.to_csv(f"{base_path}/reports/crime_by_type_summary.csv", index=False)

crime_by_type.head(10)

,primary_type,total_incidents
28,THEFT,11714
2,BATTERY,8914
5,CRIMINAL DAMAGE,5797
1,ASSAULT,4362
16,MOTOR VEHICLE THEFT,4081
21,OTHER OFFENSE,3449
8,DECEPTIVE PRACTICE,2833
3,BURGLARY,2443
17,NARCOTICS,1391
7,CRIMINAL TRESPASS,1202


In [18]:
# =========================================
# STEP 18: ARRESTS BY DISTRICT SUMMARY
# =========================================

if "district" in arrests.columns:
    arrests_by_district = arrests.groupby("district").size().reset_index(name="total_arrests").sort_values("total_arrests", ascending=False)
else:
    arrests_by_district = pd.DataFrame()

arrests_by_district.to_csv(f"{base_path}/reports/arrests_by_district_summary.csv", index=False)

arrests_by_district.head(10)

""


In [19]:
# =========================================
# STEP 19: JAIL UTILIZATION SUMMARY
# =========================================

jail_utilization_summary = jail_operations.groupby("facility_name").agg(
    avg_jail_population=("jail_population", "mean"),
    avg_capacity=("capacity", "mean"),
    avg_capacity_utilization_pct=("capacity_utilization_pct", "mean"),
    total_bookings=("bookings", "sum"),
    total_releases=("releases", "sum"),
    total_critical_incidents=("critical_incidents", "sum"),
    total_maintenance_requests=("maintenance_requests", "sum")
).reset_index()

jail_utilization_summary.to_csv(f"{base_path}/reports/jail_utilization_summary.csv", index=False)

jail_utilization_summary

,facility_name,avg_jail_population,avg_capacity,avg_capacity_utilization_pct,total_bookings,total_releases,total_critical_incidents,total_maintenance_requests
0,North Jail Facility,748.222222,850.0,88.026722,5243,4480,472,1280
1,South Jail Facility,617.244444,700.0,88.177944,5122,4638,419,1272


In [20]:
# =========================================
# STEP 20: RISK SUMMARY
# =========================================

risk_summary = risk_df.groupby(["facility_name", "risk_flag"]).size().reset_index(name="days_flagged")
risk_summary = risk_summary.sort_values(["facility_name", "days_flagged"], ascending=[True, False])

risk_summary.to_csv(f"{base_path}/reports/risk_summary.csv", index=False)

risk_summary

,facility_name,risk_flag,days_flagged
2,North Jail Facility,Normal,66
3,North Jail Facility,Staffing Gap,46
0,North Jail Facility,Facility Risk,34
1,North Jail Facility,Incident Spike,34
6,South Jail Facility,Normal,77
7,South Jail Facility,Staffing Gap,44
4,South Jail Facility,Facility Risk,30
5,South Jail Facility,Incident Spike,29


In [21]:
# =========================================
# STEP 21: ZIP THE PROJECT FOLDER
# =========================================

import shutil

zip_path = shutil.make_archive("/content/public_safety_project", 'zip', base_path)
print("ZIP created at:", zip_path)

ZIP created at: /content/public_safety_project.zip


In [22]:
# =========================================
# STEP 22: DOWNLOAD ZIP FILE
# =========================================

from google.colab import files
files.download("/content/public_safety_project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

rows = 50000

crime_types = [
"THEFT","BATTERY","CRIMINAL DAMAGE","ASSAULT",
"MOTOR VEHICLE THEFT","OTHER OFFENSE","DECEPTIVE PRACTICE",
"BURGLARY","NARCOTICS","CRIMINAL TRESPASS"
]

locations = [
"RESIDENCE","SIDEWALK","SMALL RETAIL STORE","PARKING LOT / GARAGE",
"DEPARTMENT STORE","RESTAURANT","OTHER (SPECIFY)",
"ALLEY","VEHICLE NON-COMMERCIAL","COMMERCIAL / BUSINESS"
]

dates = pd.date_range("2025-01-01","2025-12-31")

data = {
"id": np.arange(1,rows+1),
"incident_date": np.random.choice(dates,rows),
"primary_type": np.random.choice(crime_types,rows),
"location_description": np.random.choice(locations,rows),
"district": np.random.randint(1,12,rows),
"arrest": np.random.choice([0,1],rows,p=[0.8,0.2]),
"domestic": np.random.choice([0,1],rows,p=[0.9,0.1])
}

df = pd.DataFrame(data)

df["incident_year"] = df["incident_date"].dt.year
df["incident_month"] = df["incident_date"].dt.month
df["incident_month_name"] = df["incident_date"].dt.strftime("%B")
df["incident_weekday"] = df["incident_date"].dt.strftime("%A")

df.to_csv("crime_incidents_clean.csv",index=False)

print("Dataset generated:",df.shape)

Dataset generated: (50000, 11)
